# Visualization - INX Future Inc Employee Performance Analysis

## Objective
Create comprehensive visualizations to communicate findings:
1. Performance distribution and trends
2. Department comparisons
3. Feature relationships
4. Model performance metrics
5. Business insights visualization

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("Libraries loaded successfully!")

## 1. Load Data and Results

In [ ]:
# Load processed data
df = pd.read_csv('../../data/processed/employee_data_processed.csv')

# Load analysis results
dept_performance = pd.read_csv('../../data/processed/department_performance_summary.csv', index_col=0)
correlations = pd.read_csv('../../data/processed/feature_correlations.csv', index_col=0)

try:
    feature_importance = pd.read_csv('../../data/processed/feature_importance.csv')
except:
    feature_importance = None

print("Data loaded successfully!")
print(f"Dataset shape: {df.shape}")

## 2. Executive Dashboard - Key Metrics

In [ ]:
# Identify target and department columns
target_col = [col for col in df.columns if 'performance' in col.lower() and 'rating' in col.lower()]
target = target_col[0] if target_col else 'PerformanceRating'

dept_col = [col for col in df.columns if 'department' in col.lower()]
dept_col = dept_col[0] if dept_col else 'EmpDepartment'

# Create executive dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Performance Distribution', 'Department Performance',
                   'Top Performance Factors', 'Performance Trends'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'scatter'}]]
)

# 1. Performance Distribution
perf_dist = df[target].value_counts().sort_index()
fig.add_trace(
    go.Bar(x=perf_dist.index, y=perf_dist.values, name='Count',
           marker_color='steelblue'),
    row=1, col=1
)

# 2. Department Performance
dept_perf = df.groupby(dept_col)[target].mean().sort_values(ascending=True)
fig.add_trace(
    go.Bar(x=dept_perf.values, y=dept_perf.index, orientation='h',
           name='Avg Rating', marker_color='coral'),
    row=1, col=2
)

# 3. Top Factors
top_corr = correlations.iloc[:, 0].abs().sort_values(ascending=True).tail(10)
fig.add_trace(
    go.Bar(x=top_corr.values, y=top_corr.index, orientation='h',
           name='Correlation', marker_color='teal'),
    row=2, col=1
)

# 4. Performance by experience (if available)
if 'ExperienceYearsAtThisCompany' in df.columns:
    exp_perf = df.groupby('ExperienceYearsAtThisCompany')[target].mean()
    fig.add_trace(
        go.Scatter(x=exp_perf.index, y=exp_perf.values, mode='lines+markers',
                  name='Avg Rating', marker_color='green'),
        row=2, col=2
    )

# Update layout
fig.update_layout(
    title_text="INX Future Inc - Employee Performance Executive Dashboard",
    title_font_size=20,
    showlegend=False,
    height=900,
    template='plotly_white'
)

fig.show()
fig.write_html('../../data/processed/executive_dashboard.html')
print("Executive dashboard saved as HTML")

## 3. Department Deep Dive

In [ ]:
# Department performance analysis
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# 1. Average performance by department
dept_avg = df.groupby(dept_col)[target].mean().sort_values(ascending=False)
colors_dept = ['green' if x >= df[target].mean() else 'red' for x in dept_avg.values]
axes[0, 0].barh(dept_avg.index, dept_avg.values, color=colors_dept, edgecolor='black', alpha=0.7)
axes[0, 0].axvline(df[target].mean(), color='blue', linestyle='--', linewidth=2, label=f'Overall Avg: {df[target].mean():.2f}')
axes[0, 0].set_xlabel('Average Performance Rating', fontsize=12, fontweight='bold')
axes[0, 0].set_title('Department Performance Comparison', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(axis='x', alpha=0.3)

# 2. Employee count by department
dept_count = df[dept_col].value_counts()
axes[0, 1].bar(dept_count.index, dept_count.values, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 1].set_ylabel('Number of Employees', fontsize=12, fontweight='bold')
axes[0, 1].set_title('Employee Distribution by Department', fontsize=14, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Performance distribution by department
df_for_box = df[[dept_col, target]].copy()
sns.violinplot(data=df_for_box, x=dept_col, y=target, ax=axes[1, 0], palette='Set2')
axes[1, 0].set_xlabel('Department', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Performance Rating', fontsize=12, fontweight='bold')
axes[1, 0].set_title('Performance Distribution by Department', fontsize=14, fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. Performance composition by department
dept_perf_pct = pd.crosstab(df[dept_col], df[target], normalize='index') * 100
dept_perf_pct.plot(kind='bar', stacked=True, ax=axes[1, 1], colormap='viridis', edgecolor='white')
axes[1, 1].set_ylabel('Percentage (%)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Department', fontsize=12, fontweight='bold')
axes[1, 1].set_title('Performance Rating Composition by Department', fontsize=14, fontweight='bold')
axes[1, 1].legend(title='Rating', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../../data/processed/department_deep_dive.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Feature Analysis Visualizations

In [ ]:
# Top features correlation heatmap
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols = [col for col in numerical_cols if col != 'EmpNumber']

# Select top correlated features
top_features = correlations.iloc[:, 0].abs().sort_values(ascending=False).head(10).index.tolist()
top_features = [col for col in top_features if col in numerical_cols]

if len(top_features) > 0:
    # Create correlation matrix for top features + target
    corr_matrix = df[top_features + [target]].corr()
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Feature Correlation Matrix - Top Factors', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig('../../data/processed/feature_correlation_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()

## 5. Interactive Visualizations

In [ ]:
# Interactive scatter plot - if we have key features
if len(top_features) >= 2:
    fig = px.scatter(df, x=top_features[0], y=top_features[1], color=target,
                    size=top_features[0] if len(top_features) > 2 else None,
                    hover_data=[dept_col],
                    title=f'Performance Analysis: {top_features[0]} vs {top_features[1]}',
                    color_continuous_scale='viridis')
    
    fig.update_layout(height=600, template='plotly_white')
    fig.show()
    fig.write_html('../../data/processed/interactive_scatter.html')
    print("Interactive scatter plot saved")

In [ ]:
# Department performance comparison - interactive
fig = px.box(df, x=dept_col, y=target, color=dept_col,
             title='Performance Distribution by Department (Interactive)',
             labels={target: 'Performance Rating', dept_col: 'Department'})

fig.update_layout(height=600, showlegend=False, template='plotly_white')
fig.show()
fig.write_html('../../data/processed/interactive_dept_boxplot.html')
print("Interactive box plot saved")

## 6. Model Performance Visualization

In [ ]:
# Load model comparison results (if available from training)
# This would typically come from the training notebook
# For demonstration, we'll create a sample

# Feature importance visualization (if available)
if feature_importance is not None and not feature_importance.empty:
    plt.figure(figsize=(12, 8))
    
    top_n = 15
    top_features_imp = feature_importance.head(top_n)
    
    colors_imp = plt.cm.viridis(np.linspace(0, 1, len(top_features_imp)))
    
    plt.barh(range(len(top_features_imp)), top_features_imp['Importance'],
            color=colors_imp, edgecolor='black')
    plt.yticks(range(len(top_features_imp)), top_features_imp['Feature'])
    plt.xlabel('Feature Importance', fontsize=13, fontweight='bold')
    plt.title(f'Top {top_n} Most Important Features for Performance Prediction',
             fontsize=15, fontweight='bold', pad=20)
    plt.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(top_features_imp['Importance']):
        plt.text(v + 0.001, i, f'{v:.4f}', va='center', fontweight='bold', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('../../data/processed/model_feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Feature importance data not available")

## 7. Business Insights Summary Visualization

In [ ]:
# Create a comprehensive insights figure
fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Performance distribution
ax1 = fig.add_subplot(gs[0, 0])
perf_counts = df[target].value_counts().sort_index()
ax1.pie(perf_counts.values, labels=perf_counts.index, autopct='%1.1f%%',
        startangle=90, colors=sns.color_palette('pastel'))
ax1.set_title('Overall Performance Distribution', fontsize=12, fontweight='bold')

# 2. Department rankings
ax2 = fig.add_subplot(gs[0, 1:])
dept_perf = df.groupby(dept_col)[target].mean().sort_values(ascending=True)
colors_bar = ['#2ecc71' if x >= df[target].mean() else '#e74c3c' for x in dept_perf.values]
ax2.barh(dept_perf.index, dept_perf.values, color=colors_bar, edgecolor='black', alpha=0.8)
ax2.axvline(df[target].mean(), color='blue', linestyle='--', linewidth=2)
ax2.set_xlabel('Average Performance Rating', fontsize=11, fontweight='bold')
ax2.set_title('Department Performance Rankings', fontsize=12, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

# 3. Top 3 factors
ax3 = fig.add_subplot(gs[1, :])
top_3 = correlations.iloc[:, 0].abs().sort_values(ascending=True).tail(3)
colors_factors = ['#3498db', '#9b59b6', '#1abc9c']
bars = ax3.barh(top_3.index, top_3.values, color=colors_factors, edgecolor='black', alpha=0.8)
ax3.set_xlabel('Absolute Correlation with Performance', fontsize=11, fontweight='bold')
ax3.set_title('Top 3 Factors Affecting Employee Performance', fontsize=13, fontweight='bold')
ax3.grid(axis='x', alpha=0.3)

# Add value labels
for bar in bars:
    width = bar.get_width()
    ax3.text(width + 0.01, bar.get_y() + bar.get_height()/2,
            f'{width:.3f}', ha='left', va='center', fontweight='bold')

# 4. Key statistics
ax4 = fig.add_subplot(gs[2, 0])
ax4.axis('off')
stats_text = f"""
KEY STATISTICS
{"="*40}

Total Employees: {len(df):,}
Average Performance: {df[target].mean():.2f}
Performance Std Dev: {df[target].std():.2f}

Best Department: {dept_perf.index[-1]}
  • Avg Rating: {dept_perf.values[-1]:.2f}

Needs Improvement: {dept_perf.index[0]}
  • Avg Rating: {dept_perf.values[0]:.2f}

Performance Gap: {(dept_perf.values[-1] - dept_perf.values[0]):.2f}
"""
ax4.text(0.1, 0.5, stats_text, fontsize=11, family='monospace',
        verticalalignment='center')

# 5. Rating distribution bar
ax5 = fig.add_subplot(gs[2, 1:])
rating_dist = df[target].value_counts().sort_index()
bars2 = ax5.bar(rating_dist.index, rating_dist.values, color='steelblue',
               edgecolor='black', alpha=0.7)
ax5.set_xlabel('Performance Rating', fontsize=11, fontweight='bold')
ax5.set_ylabel('Number of Employees', fontsize=11, fontweight='bold')
ax5.set_title('Employee Count by Performance Rating', fontsize=12, fontweight='bold')
ax5.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars2:
    height = bar.get_height()
    ax5.text(bar.get_x() + bar.get_width()/2., height + 5,
            f'{int(height)}', ha='center', va='bottom', fontweight='bold')

# Overall title
fig.suptitle('INX Future Inc - Employee Performance Analysis Summary',
            fontsize=18, fontweight='bold', y=0.98)

plt.savefig('../../data/processed/comprehensive_insights_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nAll visualizations created and saved successfully!")

## 8. Export Summary

In [ ]:
print("\n" + "="*100)
print(" " * 35 + "VISUALIZATION SUMMARY")
print("="*100)

visualizations_created = [
    'executive_dashboard.html (Interactive)',
    'department_deep_dive.png',
    'feature_correlation_heatmap.png',
    'interactive_scatter.html',
    'interactive_dept_boxplot.html',
    'model_feature_importance.png',
    'comprehensive_insights_summary.png'
]

print("\nVisualizations Created:")
for i, viz in enumerate(visualizations_created, 1):
    print(f"  {i}. {viz}")

print("\nAll files saved in: data/processed/")
print("\n" + "="*100)
print("Visualization process completed successfully!")
print("="*100)